In [1]:
#Import all packages
import os
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
import contextily as ctx
import scipy.stats
import scipy.interpolate
import tqdm
from pathlib import Path
import xarray as xr
import skgstat as skg
import seaborn as sns
import pysal
from pysal.explore import esda
from pysal.lib import weights
from splot.esda import moran_scatterplot
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from matplotlib_scalebar.scalebar import ScaleBar
import rasterio
from rasterio.plot import show
import rasterio
import xarray as xr
import numpy as np
from pathlib import Path
import re
from datetime import datetime
import geopandas as gpd
import re
from datetime import datetime
import numpy as np
import xarray as xr
import rasterio
from pathlib import Path
from rasterio.features import rasterize
from shapely.geometry import mapping
from shapely.geometry import LineString, Point, MultiPoint, GeometryCollection
import shapely.ops as ops
import pandas as pd
import numpy as np
import geopandas as gpd

/opt/conda/lib/python3.11/site-packages/spaghetti/network.py:41: FutureWarning: The next major release of pysal/spaghetti (2.0.0) will drop support for all ``libpysal.cg`` geometries. This change is a first step in refactoring ``spaghetti`` that is expected to result in dramatically reduced runtimes for network instantiation and operations. Users currently requiring network and point pattern input as ``libpysal.cg`` geometries should prepare for this simply by converting to ``shapely`` geometries.
  warnings.warn(dep_msg, FutureWarning, stacklevel=1)


In [2]:
#Summon the Data
DATA = Path("/home/jovyan/Society_of_Bouy_Cowboys/Data")

NETCDF = DATA / "NET_CDF"
ICE = DATA / "Ice_edge"
SAT = DATA / "SAT_GeoTiff"

# buoy files
moor_files = {
    "S1P1": NETCDF / "S1P1.nc",
    "S1P2": NETCDF / "S1P2.nc",
    "S1P3": NETCDF / "S1P3.nc",
    "S1A1": NETCDF / "S1A1.nc"
}

In [3]:
#Create XArray for all GeoTiffs with Ice Edge

# Load all GeoTIFFs into a single xarray Dataset with a time dimension
scenes = {}
for tif_path in sorted(SAT.glob("*.tif")):
    m = re.search(r"_(\d{8})(\d{4})_", tif_path.stem)
    if m:
        t = datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M")
    else:
        continue
    with rasterio.open(tif_path) as src:
        scenes[t] = {
            "sigma0":    src.read(1),
            "png":       src.read(2),
            "bounds":    src.bounds,
            "transform": src.transform,
            "shape":     (src.height, src.width),
        }

# Parse datetime from filename e.g. ice_edge_201911260350.geojson
ice_edges = {}
for geojson_path in sorted(ICE.glob("*.geojson")):
    m = re.search(r"_(\d{8})(\d{4})", geojson_path.stem)
    if m:
        t = datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M")
        ice_edges[t] = gpd.read_file(geojson_path).to_crs("EPSG:32604")

print(f"Loaded {len(scenes)} SAR scenes")
print(f"Loaded {len(ice_edges)} ice edge files")

# ── Sort by time ───────────────────────────────────────────────────────────
times  = sorted(scenes.keys())
bounds = scenes[times[0]]["bounds"]
shape  = scenes[times[0]]["shape"]

# ── Rasterize ice edges onto the SAR grid ──────────────────────────────────
# For each SAR scene, if a matching ice edge exists (within 1 hour),
# burn it into a binary raster (1 = ice edge, 0 = no data).
# If no ice edge exists for that scene, fill with NaN.

def find_nearest_edge(t, ice_edges, max_hours=1):
    """Return the ice edge GeoDataFrame closest in time to t, within max_hours."""
    best_t, best_dt = None, None
    for et in ice_edges:
        dt = abs((t - et).total_seconds()) / 3600
        if dt <= max_hours and (best_dt is None or dt < best_dt):
            best_t, best_dt = et, dt
    return ice_edges[best_t] if best_t else None

ice_rasters = []
for t in times:
    edge_gdf = find_nearest_edge(t, ice_edges, max_hours=1)
    if edge_gdf is not None and len(edge_gdf) > 0:
        # Burn the line geometry into the raster grid
        burned = rasterize(
            [(mapping(geom), 1) for geom in edge_gdf.geometry],
            out_shape = shape,
            transform = scenes[t]["transform"],
            fill      = 0,
            dtype     = np.float32,
        )
    else:
        burned = np.full(shape, np.nan, dtype=np.float32)
    ice_rasters.append(burned)

print(f"Ice edge rasters built: {sum(np.any(r == 1) for r in ice_rasters)}/{len(times)} scenes have an ice edge")

# ── Build xarray Dataset ───────────────────────────────────────────────────
SAT_ds = xr.Dataset(
    data_vars=dict(
        SAR_backscatter=(["time", "y", "x"],
                np.stack([scenes[t]["sigma0"] for t in times]),
                {"long_name": "SAR backscatter", "units": "linear"}),
        ice_edge       =(["time", "y", "x"],
                np.stack(ice_rasters),
                {"long_name": "Ice edge (rasterized)", "units": "binary 0/1"}),
    ),
    coords=dict(
        time=(["time"], np.array(times, dtype="datetime64[ns]")),
        x   =(["x"],    np.linspace(bounds.left,   bounds.right, shape[1])),
        y   =(["y"],    np.linspace(bounds.top,     bounds.bottom, shape[0])),
    ),
    attrs=dict(crs="EPSG:32604"),
)

SAT_ds

Loaded 25 SAR scenes
Loaded 7 ice edge files
Ice edge rasters built: 8/25 scenes have an ice edge


<xarray.Dataset> Size: 801MB
Dimensions:          (time: 25, y: 2001, x: 2001)
Coordinates:
  * time             (time) datetime64[ns] 200B 2019-11-07T17:48:00 ... 2019-...
  * x                (x) float64 16kB 3.321e+05 3.321e+05 ... 4.374e+05
  * y                (y) float64 16kB 7.86e+06 7.86e+06 ... 7.755e+06 7.755e+06
Data variables:
    SAR_backscatter  (time, y, x) float32 400MB nan nan nan nan ... nan nan nan
    ice_edge         (time, y, x) float32 400MB nan nan nan nan ... nan nan nan
Attributes:
    crs:      EPSG:32604

In [4]:
def get_mooring_latlon(ncfile):
    ds = xr.open_dataset(ncfile)

    lat = np.asarray(ds["lat_lagrangian"]).astype(float).ravel()
    lon = np.asarray(ds["lon_lagrangian"]).astype(float).ravel()

    good = np.isfinite(lat) & np.isfinite(lon)

    lat = lat[good]
    lon = lon[good]

    if len(lat) == 0:
        raise ValueError(f"No valid lat/lon found in {ncfile}")

    # median gives robust representative location
    return np.median(lat), np.median(lon)

rows = []

for name, fn in moor_files.items():
    lat, lon = get_mooring_latlon(fn)
    rows.append({
        "source": name,
        "lat": lat,
        "lon": lon
    })

moor_df = pd.DataFrame(rows)

moor_gdf = gpd.GeoDataFrame(
    moor_df,
    geometry=gpd.points_from_xy(moor_df["lon"], moor_df["lat"]),
    crs="EPSG:4326"
)

# convert buoys into analysis CRS
moor_gdf = moor_gdf.to_crs("EPSG:32604")

moor_gdf

,source,lat,lon,geometry
0,S1P1,70.34613,-162.05704,POINT (385288.486 7807356.201)
1,S1P2,70.39473,-162.13675,POINT (382579.19 7812922.163)
2,S1P3,70.43995,-162.20300,POINT (380366.746 7818088.385)
3,S1A1,70.48695,-162.28278,POINT (377672.411 7823482.253)


In [5]:
#For Each Mooring Site, create XArray 
S1P1_ds= xr.open_dataset('/home/jovyan/Society_of_Bouy_Cowboys/Data/NET_CDF/S1P1.nc')

S1P2_ds= xr.open_dataset('/home/jovyan/Society_of_Bouy_Cowboys/Data/NET_CDF/S1P2.nc')

S1P3_ds= xr.open_dataset('/home/jovyan/Society_of_Bouy_Cowboys/Data/NET_CDF/S1P3.nc')

S1P4_ds= xr.open_dataset('/home/jovyan/Society_of_Bouy_Cowboys/Data/NET_CDF/S1P4.nc')

S1A1_ds= xr.open_dataset('/home/jovyan/Society_of_Bouy_Cowboys/Data/NET_CDF/S1A1.nc')

ds_list = [S1P1_ds, S1P2_ds, S1P3_ds, S1A1_ds]

In [6]:
xr.concat(ds_list, dim='source')

<xarray.Dataset> Size: 43MB
Dimensions:         (source: 4, time: 2604, freq: 84)
Coordinates:
  * time            (time) datetime64[ns] 21kB 2019-11-09T07:27:07.000002816 ...
  * freq            (freq) float64 672B 0.009766 0.009766 ... 0.4902 0.4902
Dimensions without coordinates: source
Data variables: (12/13)
    lat_lagrangian  (source, time) float64 83kB nan 70.35 nan ... nan 70.49
    lon_lagrangian  (source, time) float64 83kB nan -162.1 nan ... nan -162.3
    watertemp       (source, time) float64 83kB nan 20.76 nan ... nan nan nan
    sigwaveheight   (source, time) float64 83kB nan nan nan ... 1.709 nan 1.962
    peakwaveperiod  (source, time) float64 83kB nan nan nan ... 7.211 nan 7.211
    peakwavedirT    (source, time) float64 83kB nan 9.999e+03 nan ... nan nan
    ...              ...
    a1              (source, freq, time) float64 7MB nan nan nan ... nan nan nan
    b1              (source, freq, time) float64 7MB nan nan nan ... nan nan nan
    a2              (source, freq, time) float64 7MB nan nan nan ... nan nan nan
    b2              (source, freq, time) float64 7MB nan nan nan ... nan nan nan
    check           (source, freq, time) float64 7MB nan nan nan ... nan nan nan
    depth           (source, time) float64 83kB nan 13.0 nan nan ... nan nan nan

In [7]:
S1P3_ds.time

<xarray.DataArray 'time' (time: 749)> Size: 6kB
array(['2019-11-09T07:30:50.000000256', '2019-11-09T08:00:50.000003840',
       '2019-11-09T08:30:49.999996928', ..., '2019-11-24T20:30:49.999996928',
       '2019-11-24T21:00:50.000000256', '2019-11-24T21:30:50.000003840'],
      dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 6kB 2019-11-09T07:30:50.000000256 ... 2019...
Attributes:
    long_name:      time
    standard_name:  time

In [8]:
common_time = pd.date_range(
    S1A1_ds.time.min().values,
    S1A1_ds.time.max().values,
    freq="30min"
)

S1A1_ds_halfhour = S1A1_ds.interp(time=common_time)
S1A1_ds_halfhour

<xarray.Dataset> Size: 1MB
Dimensions:         (time: 725, freq: 42)
Coordinates:
  * freq            (freq) float64 336B 0.009766 0.02148 ... 0.4785 0.4902
  * time            (time) datetime64[ns] 6kB 2019-11-09T20:17:13.250987264 ....
Data variables:
    lat_lagrangian  (time) float64 6kB 70.49 70.49 70.49 ... 70.49 70.49 70.49
    lon_lagrangian  (time) float64 6kB -162.3 -162.3 -162.3 ... -162.3 -162.3
    sigwaveheight   (time) float64 6kB 0.656 0.6704 0.6849 ... 1.709 1.836 1.962
    peakwaveperiod  (time) float64 6kB 7.877 7.544 7.211 ... 7.211 7.211 7.211
    peakwavedirT    (time) float64 6kB nan nan nan nan nan ... nan nan nan nan
    energy          (freq, time) float64 244kB 0.0 0.0 0.0 ... 0.03249 0.04849
    a1              (freq, time) float64 244kB nan nan nan nan ... nan nan nan
    b1              (freq, time) float64 244kB nan nan nan nan ... nan nan nan
    a2              (freq, time) float64 244kB nan nan nan nan ... nan nan nan
    b2              (freq, time) float64 244kB nan nan nan nan ... nan nan nan
    check           (freq, time) float64 244kB nan nan nan nan ... nan nan nan

In [9]:
ds_list = [S1P1_ds, S1P2_ds, S1P3_ds, S1A1_ds_halfhour]
labels = ["S1P1", "S1P2", "S1P3", "S1A1"]

# common 30-minute grid over the overlapping time range
tmin = max(pd.Timestamp(ds.time.min().values).ceil("30min") for ds in ds_list)
tmax = min(pd.Timestamp(ds.time.max().values).floor("30min") for ds in ds_list)
common_time = pd.date_range(tmin, tmax, freq="30min")

# snap each dataset to the shared grid
aligned = [
    ds.reindex(time=common_time, method="nearest", tolerance=pd.Timedelta("15min"))
    for ds in ds_list
]

# combine
ds_out = xr.concat(
    aligned,
    dim=xr.DataArray(labels, dims="source", name="source")
)
ds_out

<xarray.Dataset> Size: 12MB
Dimensions:         (source: 4, time: 718, freq: 84)
Coordinates:
  * time            (time) datetime64[ns] 6kB 2019-11-09T20:30:00 ... 2019-11...
  * freq            (freq) float64 672B 0.009766 0.009766 ... 0.4902 0.4902
  * source          (source) <U4 64B 'S1P1' 'S1P2' 'S1P3' 'S1A1'
Data variables: (12/13)
    lat_lagrangian  (source, time) float64 23kB 70.35 70.35 ... 70.49 70.49
    lon_lagrangian  (source, time) float64 23kB -162.1 -162.1 ... -162.3 -162.3
    watertemp       (source, time) float64 23kB 15.82 15.73 14.57 ... nan nan
    sigwaveheight   (source, time) float64 23kB nan nan nan ... 1.89 1.827 1.823
    peakwaveperiod  (source, time) float64 23kB nan nan nan ... 7.211 7.544
    peakwavedirT    (source, time) float64 23kB nan nan nan nan ... nan nan nan
    ...              ...
    a1              (source, freq, time) float64 2MB nan nan nan ... nan nan nan
    b1              (source, freq, time) float64 2MB nan nan nan ... nan nan nan
    a2              (source, freq, time) float64 2MB nan nan nan ... nan nan nan
    b2              (source, freq, time) float64 2MB nan nan nan ... nan nan nan
    check           (source, freq, time) float64 2MB nan nan nan ... nan nan nan
    depth           (source, time) float64 23kB 13.0 13.0 13.0 ... nan nan nan

In [10]:
# ---------------------------------------------------
# A. Sort moorings and define a 1D transect
# ---------------------------------------------------
# If you want a specific mooring to be "0 point", set it here:
ZERO_MOORING = "S1P1"

# Keep your mooring order explicit
moor_order = ["S1P1", "S1P2", "S1P3", "S1A1"]
moor_gdf = moor_gdf.set_index("source").loc[moor_order].reset_index()

# Build a line through the moorings in order
transect = LineString(moor_gdf.geometry.tolist())

# Distance along transect from the 0-point
zero_point = moor_gdf.loc[moor_gdf["source"] == ZERO_MOORING, "geometry"].iloc[0]
s0 = transect.project(zero_point)

moor_gdf["s_m"] = moor_gdf.geometry.apply(lambda geom: transect.project(geom) - s0)
moor_gdf["s_km"] = moor_gdf["s_m"] / 1000.0

print(moor_gdf[["source", "lat", "lon", "s_m", "s_km"]])

  source       lat        lon           s_m       s_km
0   S1P1  70.34613 -162.05704      0.000000   0.000000
1   S1P2  70.39473 -162.13675   6190.332733   6.190333
2   S1P3  70.43995 -162.20300  11810.364732  11.810365
3   S1A1  70.48695 -162.28278  17839.730471  17.839730


In [11]:
# ---------------------------------------------------
# B. Helper functions for ice-edge intersection
# ---------------------------------------------------
def _extract_points_from_intersection(intersection):
    """
    Convert shapely intersection output into a list of Points.
    Handles Point, MultiPoint, LineString, MultiLineString, GeometryCollection.
    """
    pts = []

    if intersection.is_empty:
        return pts

    geom_type = intersection.geom_type

    if geom_type == "Point":
        pts = [intersection]

    elif geom_type == "MultiPoint":
        pts = list(intersection.geoms)

    elif geom_type == "LineString":
        # If the edge overlaps the transect for a segment, use midpoint of overlap
        pts = [intersection.interpolate(0.5, normalized=True)]

    elif geom_type == "MultiLineString":
        pts = [g.interpolate(0.5, normalized=True) for g in intersection.geoms]

    elif geom_type == "GeometryCollection":
        for g in intersection.geoms:
            pts.extend(_extract_points_from_intersection(g))

    return pts


def ice_edge_distance_along_transect(edge_gdf, transect, s0, moor_s_values=None):
    """
    Return ice-edge distance along transect in meters relative to the 0-point.
    If multiple crossings exist, choose the one nearest the mooring array center.
    """
    if edge_gdf is None or len(edge_gdf) == 0:
        return np.nan

    # Make sure CRS matches the transect CRS
    if edge_gdf.crs is None:
        raise ValueError("Ice edge GeoDataFrame has no CRS.")
    if edge_gdf.crs.to_string() != "EPSG:32604":
        edge_gdf = edge_gdf.to_crs("EPSG:32604")

    edge_union = edge_gdf.geometry.union_all() if hasattr(edge_gdf.geometry, "union_all") else edge_gdf.unary_union
    inter = transect.intersection(edge_union)

    pts = _extract_points_from_intersection(inter)
    if len(pts) == 0:
        return np.nan

    s_candidates = np.array([transect.project(pt) - s0 for pt in pts])

    # If there are multiple crossings, choose the one closest to the mooring-array center
    if moor_s_values is None:
        target = 0.0
    else:
        target = np.nanmean(moor_s_values)

    return s_candidates[np.argmin(np.abs(s_candidates - target))]

In [12]:
# ---------------------------------------------------
# C. Build ice-edge distance table
# ---------------------------------------------------
# Convert SAT_ds time values to pandas timestamps
time_index = pd.to_datetime(SAT_ds.time.values)

ice_rows = []
moor_s_values = moor_gdf["s_m"].values

for t in time_index:
    # Match to your existing dict keys
    t_py = pd.Timestamp(t).to_pydatetime()

    # Reuse your nearest-edge logic if you want:
    edge_gdf = find_nearest_edge(t_py, ice_edges, max_hours=1)

    if edge_gdf is None:
        ice_s_m = np.nan
        has_edge = False
    else:
        ice_s_m = ice_edge_distance_along_transect(
            edge_gdf=edge_gdf,
            transect=transect,
            s0=s0,
            moor_s_values=moor_s_values
        )
        has_edge = np.isfinite(ice_s_m)

    ice_rows.append({
        "time": t,
        "ice_edge_s_m": ice_s_m,
        "ice_edge_s_km": ice_s_m / 1000.0 if np.isfinite(ice_s_m) else np.nan,
        "has_ice_edge": has_edge
    })

ice_df = pd.DataFrame(ice_rows)
ice_df

,time,ice_edge_s_m,ice_edge_s_km,has_ice_edge
0,2019-11-07 17:48:00,NaN,NaN,False
1,2019-11-08 04:15:00,NaN,NaN,False
2,2019-11-08 04:19:00,NaN,NaN,False
3,2019-11-08 17:40:00,NaN,NaN,False
4,2019-11-09 03:46:00,NaN,NaN,False
5,2019-11-11 04:27:00,NaN,NaN,False
6,2019-11-14 17:40:00,NaN,NaN,False
7,2019-11-15 04:11:00,NaN,NaN,False
8,2019-11-16 03:41:00,8511.028193,8.511028,True
9,2019-11-16 17:23:00,NaN,NaN,False


In [16]:
for name, fn in moor_files.items():
    ds = xr.open_dataset(fn)
    print(f"{name} variables:", list(ds.data_vars))
    print(f"{name} coords:", list(ds.coords))
    break

S1P1 variables: ['lat_lagrangian', 'lon_lagrangian', 'watertemp', 'sigwaveheight', 'peakwaveperiod', 'peakwavedirT', 'energy', 'a1', 'b1', 'a2', 'b2', 'check', 'depth']
S1P1 coords: ['time', 'freq']


In [20]:
import xarray as xr

def find_wave_height_var(ds):
    candidates = [
        "Hs", "hs", "Hm0", "hm0", "wave_height", "sig_wave_height",
        "significant_wave_height", "Hsig"
    ]
    for v in candidates:
        if v in ds.data_vars:
            return v
    raise KeyError(f"No wave-height variable found. Available variables: {list(ds.data_vars)}")


def mooring_timeseries_to_df(name, ncfile):
    ds = xr.open_dataset(ncfile)

    hs_var = find_wave_height_var(ds)
    da = ds[hs_var]

    # squeeze singleton dims
    da = da.squeeze()

    # must have time
    if "time" not in da.coords:
        raise ValueError(f"{name}: wave-height variable '{hs_var}' has no time coordinate.")

    df = pd.DataFrame({
        "time": pd.to_datetime(da["time"].values),
        "wave_height_m": np.asarray(da.values).astype(float).ravel()
    })

    df["source"] = name
    return df


wave_rows = []

for name, fn in moor_files.items():
    ds = xr.open_dataset(fn)

    # use the actual wave-height variable name you found
    da = ds["sigwaveheight"].squeeze()

    # make sure time exists
    if "time" not in da.coords:
        raise ValueError(f"{name}: 'sigwaveheight' has no time coordinate")

    tmp = pd.DataFrame({
        "time": pd.to_datetime(da["time"].values),
        "wave_height_m": np.asarray(da.values).ravel(),
        "source": name
    })

    wave_rows.append(tmp)

wave_df = pd.concat(wave_rows, ignore_index=True)

wave_df.head()

,time,wave_height_m,source
0,2019-11-09 07:29:13.000003584,NaN,S1P1
1,2019-11-09 07:59:12.999996672,NaN,S1P1
2,2019-11-09 08:29:13.000000000,NaN,S1P1
3,2019-11-09 08:59:13.000003584,NaN,S1P1
4,2019-11-09 09:29:12.999996672,NaN,S1P1


In [21]:
# -----------------------------
# 1. fixed mooring distances
# -----------------------------
moor_dist_df = moor_gdf[["source", "s_m", "s_km"]].copy()
moor_dist_df = moor_dist_df.rename(columns={
    "s_m": "moor_s_m",
    "s_km": "moor_s_km"
})

# -----------------------------
# 2. ice-edge distance table
# -----------------------------
# use this if you already have ice_df:
ice_df["time"] = pd.to_datetime(ice_df["time"])

# -----------------------------
# 3. mooring wave-height table
# -----------------------------
# CASE A: combined mooring xarray dataset
wave_var = "sigwaveheight"   # change as needed
wave_df = SAT_ds[[wave_var]].to_dataframe().reset_index()
wave_df = wave_df.rename(columns={wave_var: "wave_height_m"})
wave_df["time"] = pd.to_datetime(wave_df["time"])

# -----------------------------
# 4. merge mooring distances
# -----------------------------
wave_df = wave_df.merge(
    moor_dist_df,
    on="source",
    how="left"
)

# -----------------------------
# 5. merge ice-edge distances
# -----------------------------
wave_df = wave_df.merge(
    ice_df[["time", "ice_edge_s_m", "ice_edge_s_km", "has_ice_edge"]],
    on="time",
    how="left"
)

# -----------------------------
# 6. relative distance columns
# -----------------------------
wave_df["moor_to_ice_m"] = wave_df["moor_s_m"] - wave_df["ice_edge_s_m"]
wave_df["moor_to_ice_km"] = wave_df["moor_to_ice_m"] / 1000.0

wave_df.head()

KeyError: 'sigwaveheight'